In [2]:
!nvidia-smi

Wed Oct  7 02:37:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.43.02              KMD Version: 610.43.02     CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:08:00.0 Off |                  N/A |
|  0%   32C    P8             22W /  325W |      45MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import sys
!{sys.executable} -m pip install datasets huggingface_hub sentencepiece sacrebleu scipy pandas matplotlib

  Using cached datasets-5.1.0-py3-none-any.whl.metadata (24 kB)
  Using cached pyarrow-25.0.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached xxhash-4.0.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (17 kB)
  Using cached multiprocess-0.70.19-py312-none-any.whl.metadata (7.5 kB)
  Using cached huggingface_hub-2.1.1-py3-none-any.whl.metadata (16 kB)
  Using cached httpx2-2.13.1-py3-none-any.whl.metadata (9.8 kB)
  Using cached aiohttp-3.14.4-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (8.3 kB)
  Using cached httpcore2-2.13.1-py3-none-any.whl.metadata (26 kB)
  Using cached truststore-0.10.4-py3-none-any.whl.metadata (4.4 kB)
  Using cached aiohappyeyeballs-2.7.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp312-cp312-manylinux1

In [ ]:
## run download in terminal

In [11]:
ls data/raw

MultiUN.en-ar.zip  News-Commentary.en-ar.zip  TED2020.en-ka.zip
NLLB.en-ka.zip     TED2020.en-ar.zip          WikiMatrix.en-ka.zip


In [12]:
import os, re, json, random, unicodedata, zipfile
from datasets import load_dataset
os.chdir("/workspace/shreshth")
SEED = 42
ARABIC = re.compile(r"[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\uFB50-\uFDFF\uFE70-\uFEFF]")

def norm(s):
    return unicodedata.normalize("NFC", s or "").strip()

def read_moses(path):
    """Read an OPUS zip with two aligned text files (.en and .ar)."""
    with zipfile.ZipFile(path) as z:
        names = z.namelist()
        en = z.read(next(n for n in names if n.endswith(".en"))).decode("utf-8", "ignore").split("\n")
        ar = z.read(next(n for n in names if n.endswith(".ar"))).decode("utf-8", "ignore").split("\n")
    return list(zip(en, ar))

def clean(rows):
    """<=250 chars, >=3 English words, target really Arabic, sane length ratio, no duplicates."""
    out, seen = [], set()
    for en, ar in rows:
        en, ar = norm(en), norm(ar)
        if not en or not ar or len(en) > 250 or len(ar) > 250 or len(en.split()) < 3:
            continue
        if not ARABIC.search(ar) or ar == en:
            continue
        r = len(ar) / len(en)
        if r < 0.4 or r > 2.0 or (en, ar) in seen:
            continue
        seen.add((en, ar))
        out.append({"en": en, "tgt": ar})
    return out

raw = {}
try:
    ds = load_dataset("Helsinki-NLP/opus-100", "ar-en", split="train")
    raw["OPUS-100"] = [(ex["translation"]["en"], ex["translation"]["ar"]) for ex in ds]
except Exception as e:
    print(f"OPUS-100 unavailable: {type(e).__name__}: {e}")
for name in ["TED2020", "News-Commentary", "MultiUN"]:
    path = f"data/raw/{name}.en-ar.zip"
    try:
        raw[name] = read_moses(path)
    except Exception as e:
        print(f"{name} unavailable: {type(e).__name__}: {e}")

cleaned = {k: clean(v) for k, v in raw.items()}
for k in raw:
    print(f"{k:16s}: {len(raw[k]):>11,} raw -> {len(cleaned[k]):>11,} after cleaning")

ar-en/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  214kB            

ar-en/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

ar-en/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 99.3MB            

ar-en/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

ar-en/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  979kB            

ar-en/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

OPUS-100        :   1,000,000 raw ->     791,046 after cleaning
TED2020         :     407,596 raw ->     379,263 after cleaning
News-Commentary :      97,385 raw ->      14,905 after cleaning
MultiUN         :   9,759,126 raw ->   6,018,459 after cleaning


In [13]:
rng = random.Random(1)
for k, pairs in cleaned.items():
    print(f"\n======== {k} ========")
    for p in rng.sample(pairs, min(5, len(pairs))):
        print("EN:", p["en"]); print("AR:", p["tgt"]); print()


======== OPUS-100 ========
EN: Need to speak to ezra face-to-face.
AR: -أحتاجُ أن أُواجه (إيزرا) و أكلّمه

EN: List of issues (continued)
AR: قائمة القضايا (تابع)

EN: You reminded me, and...
AR: لقد نسيت. ذكرتني, و

EN: Mnh-mnh. It almost killed me.
AR: لقد كانت تقتلني تقريباً

EN: - Peter, you're so dramatic!
AR: - "بيتر"، تتصرف بدرامية.


======== TED2020 ========
EN: And we thought we'd probably have to look at several hundred before we got the picture. But as you can see, there's seven out of 20 had a change which was very high.
AR: وأعتقدنا بأننا بحاجة لفحص عدة مئات قبل أن نحصل على صورة عن الوضع. لكن كما ترون، يوجد 7 من أصل 20 كان لديهم تغير، والذي كان كبيراً جداً.

EN: And I wish there were more people like you, because I'd wager to guess there's more people in the room living in a similar situation that might not know their neighbors.
AR: وأتمنى أن يكون هناك الكثير من الناس مثلك، لأني أراهن أن كثير من الناس في الغرفة يعيشون في موقف مشابه ولا يعرفون على الأغلب جيرانهم.

EN: Tho

In [18]:
from huggingface_hub import login
login()

In [19]:
for split in ["dev", "devtest"]:
    ds = load_dataset("openlanguagedata/flores_plus", "arb_Arab", split=split).sort("id")
    en_ids = [json.loads(l)["id"] for l in open(f"data/flores/{split}.en.jsonl")]
    assert list(ds["id"]) == en_ids, "FLORES ids don't line up with English"
    with open(f"data/flores/{split}.ar.jsonl", "w", encoding="utf-8") as f:
        for i, t in zip(ds["id"], ds["text"]):
            f.write(json.dumps({"id": i, "text": norm(t)}, ensure_ascii=False) + "\n")
    print(f"FLORES {split}: {len(ds)} Arabic sentences saved")

Resolving data files:   0%|          | 0/228 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/222 [00:00<?, ?it/s]

arb_Arab.jsonl:   0%|          | 0.00/498k [00:00<?, ?B/s]

arb_Arab.jsonl:   0%|          | 0.00/519k [00:00<?, ?B/s]

Generating dev split:   0%|          | 0/997 [00:00<?, ? examples/s]

...

/venv/main/lib/python3.12/site-packages/pyarrow/compute.py:230: FutureWarning: Specifying null_placement in SortOptions is deprecated as of 25.0.0. Specify null_placement per sort_key instead.
  return options_class(*args, **kwargs)


FLORES dev: 997 Arabic sentences saved


Resolving data files:   0%|          | 0/228 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/222 [00:00<?, ?it/s]

FLORES devtest: 1012 Arabic sentences saved


In [20]:
LEADING_NUM = re.compile(r"^\s*[\(\[]?\d+[\)\]]?\s*[-–.]\s*")     # "25- ", "6 - ", "(3) - "

def strip_num(p):
    en, ar = p["en"], p["tgt"]
    if LEADING_NUM.match(ar) and not LEADING_NUM.match(en):   # number only on the Arabic side
        ar = LEADING_NUM.sub("", ar)
    return {"en": en, "tgt": ar}

TARGET, UN_CAP = 1_000_000, 400_000
rng = random.Random(SEED)

pairs, seen = [], set()
def add(rows, name, limit=None):
    rows = [strip_num(p) for p in rows]
    rng.shuffle(rows)
    n = 0
    for p in rows:
        if limit is not None and n >= limit:
            break
        key = (p["en"], p["tgt"])
        if key in seen:
            continue
        seen.add(key); pairs.append(p); n += 1
    print(f"  {name:16s}: {n:,} pairs added")

for name in ["TED2020", "News-Commentary", "OPUS-100"]:
    add(cleaned.get(name, []), name)
add(cleaned.get("MultiUN", []), "MultiUN", limit=max(0, min(UN_CAP, TARGET - len(pairs))))

rng.shuffle(pairs)
pairs = pairs[:TARGET]
os.makedirs("data/en-ar", exist_ok=True)
for name, part in [("val", pairs[:2000]), ("train", pairs[2000:])]:
    with open(f"data/en-ar/{name}.jsonl", "w", encoding="utf-8") as f:
        for p in part:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")
print(f"saved {len(pairs) - 2000:,} train / 2,000 val to data/en-ar/")

  TED2020         : 379,263 pairs added
  News-Commentary : 14,905 pairs added
  OPUS-100        : 788,405 pairs added
  MultiUN         : 0 pairs added
saved 998,000 train / 2,000 val to data/en-ar/
